# PRAICP-1009-101 — Food Classes Classification
### DataMites™ Project Mentoring PR-0019

**Business Case:** Develop a machine learning model that can classify the type of food.

**Goal:** Build a CNN-based image classifier that identifies the correct food category
from an image, using the Food-101 dataset (101 food classes, 1,000 images per class).

**Author:** _[Your Name]_
**Run environment:** Google Colab (GPU runtime recommended: Runtime → Change runtime type → T4 GPU)

---
This notebook is fully self-contained — run every cell top to bottom on Colab and it will:
1. Download and extract the Food-101 dataset
2. Explore and visualize the data
3. Build a transfer-learning CNN model
4. Train, evaluate, and visualize results
5. Save the trained model

> **Note:** Training on all 101 classes takes 1–3 hours on a free Colab T4 GPU depending on
> epochs chosen. A `QUICK_TEST` flag is provided below to first sanity-check the whole
> pipeline on a small subset of classes in a few minutes before committing to a full run.


## 1. Setup & Imports

In [ ]:
!pip install -q tensorflow matplotlib seaborn scikit-learn pillow


In [ ]:
import os, shutil, random, pathlib, zipfile, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. Download the Dataset

Dataset source (provided in the project brief):
`https://d3ilbtxij3aepc.cloudfront.net/projects/CNN-PROJECT-1-6/food-101.zip`

This is a ~5GB zip. On Colab this typically takes a few minutes depending on network speed.


In [ ]:
DATA_DIR = pathlib.Path("/content/food-101")

if not DATA_DIR.exists():
    print("Downloading dataset...")
    !wget -q --show-progress -O /content/food-101.zip https://d3ilbtxij3aepc.cloudfront.net/projects/CNN-PROJECT-1-6/food-101.zip
    print("Extracting...")
    !unzip -q /content/food-101.zip -d /content/
    print("Done.")
else:
    print("Dataset already present.")

# NOTE: the official Food-101 zip usually extracts to a folder structure like:
#   food-101/images/<class_name>/<image>.jpg
#   food-101/meta/classes.txt, train.txt, test.txt
# Adjust IMAGES_DIR below if the extracted folder name differs.
IMAGES_DIR = pathlib.Path("/content/food-101/images")
META_DIR = pathlib.Path("/content/food-101/meta")
print("Classes found:", len(os.listdir(IMAGES_DIR)))


## 3. Configuration

Set `QUICK_TEST = True` the first time you run this notebook to verify the whole
pipeline works end-to-end in a few minutes on a small subset of classes. Once
confirmed, set it to `False` to train on the full 101-class dataset for your
actual submission results.


In [ ]:
QUICK_TEST = True          # <-- set False for the full run you'll submit
N_QUICK_CLASSES = 10       # only used when QUICK_TEST is True

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 5             # training just the new classification head
EPOCHS_FINETUNE = 5         # fine-tuning the base model afterwards
VALIDATION_SPLIT = 0.2

all_classes = sorted(os.listdir(IMAGES_DIR))
if QUICK_TEST:
    classes = all_classes[:N_QUICK_CLASSES]
else:
    classes = all_classes

print(f"Using {len(classes)} classes: {classes[:5]}{' ...' if len(classes) > 5 else ''}")


## 4. Data Exploration

In [ ]:
counts = {c: len(os.listdir(IMAGES_DIR / c)) for c in classes}
pd.Series(counts).sort_values(ascending=False).plot(kind="bar", figsize=(14,4), title="Images per class")
plt.ylabel("Image count")
plt.tight_layout()
plt.show()

print("Total images in scope:", sum(counts.values()))


In [ ]:
# Visualize a handful of sample images
fig, axes = plt.subplots(2, 5, figsize=(15,6))
sample_classes = random.sample(classes, min(10, len(classes)))
for ax, cls in zip(axes.flatten(), sample_classes):
    img_path = IMAGES_DIR / cls / random.choice(os.listdir(IMAGES_DIR / cls))
    img = keras.utils.load_img(img_path, target_size=IMG_SIZE)
    ax.imshow(img)
    ax.set_title(cls, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Build a Working Directory With Only the Selected Classes

(Skips this copy step and uses the full folder directly when `QUICK_TEST = False`
and all classes are selected — copying is only needed to isolate the subset.)


In [ ]:
WORK_DIR = pathlib.Path("/content/food_subset")

if QUICK_TEST:
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True)
    for c in classes:
        shutil.copytree(IMAGES_DIR / c, WORK_DIR / c)
    DATASET_DIR = WORK_DIR
else:
    DATASET_DIR = IMAGES_DIR

print("Dataset directory in use:", DATASET_DIR)


## 6. Train / Validation Split & Data Pipelines

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

val_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes:", NUM_CLASSES)

# Performance: cache + prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


In [ ]:
# Data augmentation applied only during training
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")


## 7. Model Architecture — Transfer Learning

We use **EfficientNetB0** pretrained on ImageNet as a frozen feature extractor,
with a custom classification head on top. This is a standard, efficient approach
for image classification tasks with limited training time/compute, and performs
strongly on food-image benchmarks.


In [ ]:
base_model = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)
base_model.trainable = False   # freeze base for the first training phase

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 8. Phase 1 — Train the Classification Head

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2),
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)


## 9. Phase 2 — Fine-Tune the Base Model

Unfreeze the top layers of EfficientNetB0 and continue training at a much lower
learning rate. This lets the network adapt its higher-level features specifically
to food images, typically giving a meaningful accuracy boost over the frozen-base model.


In [ ]:
base_model.trainable = True

# Freeze all but the last ~30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINETUNE,
    callbacks=callbacks
)


## 10. Training Curves

In [ ]:
def plot_history(h1, h2=None):
    acc = h1.history['accuracy'] + (h2.history['accuracy'] if h2 else [])
    val_acc = h1.history['val_accuracy'] + (h2.history['val_accuracy'] if h2 else [])
    loss = h1.history['loss'] + (h2.history['loss'] if h2 else [])
    val_loss = h1.history['val_loss'] + (h2.history['val_loss'] if h2 else [])

    fig, axes = plt.subplots(1, 2, figsize=(14,5))
    axes[0].plot(acc, label='Train Accuracy')
    axes[0].plot(val_acc, label='Val Accuracy')
    axes[0].set_title('Accuracy over epochs'); axes[0].legend(); axes[0].set_xlabel('Epoch')

    axes[1].plot(loss, label='Train Loss')
    axes[1].plot(val_loss, label='Val Loss')
    axes[1].set_title('Loss over epochs'); axes[1].legend(); axes[1].set_xlabel('Epoch')
    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=150)
    plt.show()

plot_history(history_head, history_finetune)


## 11. Evaluation on Validation Set

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"Final Validation Accuracy: {val_acc:.4f}")
print(f"Final Validation Loss: {val_loss:.4f}")


In [ ]:
# Confusion matrix + classification report
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(min(20, len(class_names)*0.6), min(18, len(class_names)*0.6)))
sns.heatmap(cm, annot=len(class_names) <= 15, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()


## 12. Sample Predictions

In [ ]:
plt.figure(figsize=(15, 10))
for images, labels in val_ds.take(1):
    preds = model.predict(images, verbose=0)
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        true_label = class_names[np.argmax(labels[i])]
        pred_label = class_names[np.argmax(preds[i])]
        color = "green" if true_label == pred_label else "red"
        plt.title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=9)
        plt.axis("off")
plt.tight_layout()
plt.savefig('/content/sample_predictions.png', dpi=150)
plt.show()


## 13. Save the Trained Model

In [ ]:
model.save('/content/food_classifier_model.keras')
print("Model saved.")


## 14. Conclusion

_Fill this in after your full run (`QUICK_TEST = False`):_

- Final validation accuracy achieved: **____**
- Number of classes trained on: **____**
- Observations on which food classes were most/least confused (see confusion matrix)
- Possible next steps: more epochs, different backbone (ResNet50 / EfficientNetB3),
  more aggressive augmentation, larger image size, ensembling.
